# Module 7 — DAFM-3C BCI-IV-2a LOSO (22×640)

**Final candidate architecture:** Dual Attentive Fusion Model (DAFM)-style spatial-temporal attention + EEGNet-style classifier.

The original DAFM paper reports 82.09% average accuracy on BCI Competition IV-2a and describes an interactive spatial-temporal attention module followed by an EEGNet-inherited CNN classifier. This notebook adapts that design to the project's frozen 3-class `(N,22,640)` cache. The published percentage is a benchmark, not a guarantee on this resampled cache.

**Classes:** left / right / feet  
**Outer evaluation:** BCI-IV-2a S01–S09 LOSO  
**Input:** `(N,22,640)` at 160 Hz  
**No:** DANN, MMD, CORAL, Transformer, FBCSP, target-label tuning.

### Evaluation policy

For each target subject:
- the target subject is excluded from supervised training;
- the remaining 8 subjects are the source pool;
- 20% of source trials are used for validation using stratified sampling;
- normalization is fitted on source-training trials only;
- the target labels are used only for the final test metrics.

The optional 2-seed ensemble averages class probabilities from independently trained models. It does not use test labels for selection.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / REPRODUCIBILITY
# ============================================================

from __future__ import annotations

import os
import gc
import time
import copy
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Device:", device)
print("Seed:", SEED)

Device: cpu
Seed: 42


In [2]:
# ============================================================
# CELL 2 — EXISTING PROJECT CACHE
# ============================================================

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

PROJECT_DIR = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

CACHE_DIR = (
    PROJECT_DIR
    / "cache"
)

RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "module_7_dafm"
)

FIG_DIR = (
    PROJECT_DIR
    / "figures"
    / "module_7_dafm"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_CACHE_PATH = (
    CACHE_DIR
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

if not FINAL_CACHE_PATH.exists():

    candidates = sorted(
        CACHE_DIR.glob("*.h5")
    )

    preferred = [
        p for p in candidates
        if (
            "160hz" in p.name.lower()
            or "module_5" in p.name.lower()
        )
    ]

    if not preferred:
        raise FileNotFoundError(
            f"Cannot find HDF5 cache in {CACHE_DIR}"
        )

    FINAL_CACHE_PATH = preferred[0]

print(
    "Cache:",
    FINAL_CACHE_PATH,
)

assert FINAL_CACHE_PATH.exists()

Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — CACHE METADATA / DATASET CONTRACT
# ============================================================

def decode(v):
    if isinstance(v, bytes):
        return v.decode("utf-8")
    return str(v)

with h5py.File(
    FINAL_CACHE_PATH,
    "r",
) as h5:

    X_shape = tuple(
        h5["X"].shape
    )

    X_dtype = str(
        h5["X"].dtype
    )

    metadata = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:

        metadata[key] = [
            decode(v)
            for v in h5[
                "metadata"
            ][key][:]
        ]

cache_meta_df = pd.DataFrame(
    metadata
)

cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(
        len(cache_meta_df),
        dtype=np.int64,
    ),
)

assert X_shape[1:] == (
    22,
    640,
)

assert X_dtype == "float32"

CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    c: i
    for i, c in enumerate(CLASSES)
}

ID_TO_CLASS = {
    i: c
    for c, i in CLASS_TO_ID.items()
}

N_CLASSES = 3

bci_meta = cache_meta_df[
    cache_meta_df[
        "dataset"
    ].astype(str)
    == "BCI-IV-2a"
].copy()

bci_meta["subject"] = (
    bci_meta["subject"]
    .astype(str)
)

BCI_SUBJECTS = sorted(
    bci_meta[
        "subject"
    ].unique()
)

assert len(
    BCI_SUBJECTS
) == 9

print(
    "Cache:",
    X_shape,
)

print(
    "BCI subjects:",
    BCI_SUBJECTS,
)

print(
    "BCI epochs:",
    len(bci_meta),
)

display(
    bci_meta.groupby(
        "subject"
    )[
        "harmonized_class"
    ]
    .value_counts()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=BCI_SUBJECTS,
        columns=CLASSES,
        fill_value=0,
    )
)

Cache: (9316, 22, 640)
BCI subjects: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']
BCI epochs: 1944


harmonized_class,left,right,feet
subject,,,
S01,72,72,72
S02,72,72,72
S03,72,72,72
S04,72,72,72
S05,72,72,72
S06,72,72,72
S07,72,72,72
S08,72,72,72
S09,72,72,72


In [4]:
# ============================================================
# CELL 4 — HDF5 LOADER + NUMERICAL QA
# ============================================================

def load_indices(indices):

    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    with h5py.File(
        FINAL_CACHE_PATH,
        "r",
    ) as h5:

        X = np.asarray(
            h5["X"][indices],
            dtype=np.float32,
        )

    return X


qa_idx = np.arange(
    min(512, X_shape[0]),
    dtype=np.int64,
)

X_qa = load_indices(
    qa_idx
)

nonfinite = int(
    (
        ~np.isfinite(
            X_qa
        )
    ).sum()
)

zero_var = int(
    (
        np.var(
            X_qa,
            axis=(1,2),
        )
        <= 1e-12
    ).sum()
)

print(
    "Non-finite:",
    nonfinite,
)

print(
    "Zero-variance:",
    zero_var,
)

assert nonfinite == 0

print(
    "✅ Numerical QA passed."
)

Non-finite: 0
Zero-variance: 0
✅ Numerical QA passed.


In [5]:
# ============================================================
# CELL 5 — SOURCE-ONLY ROBUST NORMALIZATION
# ============================================================

class SourceRobustNormalizer:

    def __init__(
        self,
        eps=1e-6,
    ):

        self.eps = eps
        self.median_ = None
        self.iqr_ = None

    def fit(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        values = (
            X
            .transpose(
                1,
                0,
                2,
            )
            .reshape(
                X.shape[1],
                -1,
            )
        )

        self.median_ = np.median(
            values,
            axis=1,
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(
        self,
        X,
    ):

        if self.median_ is None:
            raise RuntimeError(
                "Normalizer not fitted."
            )

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        Z = (
            X
            - self.median_[
                None, :, None
            ]
        )

        Z = (
            Z
            / (
                self.iqr_[
                    None, :, None
                ]
                + self.eps
            )
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z.astype(
            np.float32
        )

print(
    "✅ Robust source normalizer ready."
)

✅ Robust source normalizer ready.


In [6]:
# ============================================================
# CELL 6 — STRATIFIED SOURCE TRIAL SPLIT
# ============================================================

def stratified_split(
    n,
    y,
    val_fraction=0.20,
    seed=SEED,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    rng = np.random.default_rng(
        seed
    )

    train_parts = []
    val_parts = []

    all_idx = np.arange(
        n,
        dtype=np.int64,
    )

    for cls in range(
        N_CLASSES
    ):

        idx = all_idx[
            y == cls
        ].copy()

        rng.shuffle(
            idx
        )

        n_val = max(
            1,
            int(
                round(
                    len(idx)
                    * val_fraction
                )
            ),
        )

        val_parts.append(
            idx[:n_val]
        )

        train_parts.append(
            idx[n_val:]
        )

    train_idx = np.concatenate(
        train_parts
    )

    val_idx = np.concatenate(
        val_parts
    )

    rng.shuffle(
        train_idx
    )

    rng.shuffle(
        val_idx
    )

    return (
        train_idx,
        val_idx,
    )


print(
    "✅ Stratified split ready."
)

✅ Stratified split ready.


In [7]:
# ============================================================
# CELL 7 — DAFM SPATIAL-TEMPORAL ATTENTION
# ============================================================

class DAFM3C(
    nn.Module
):
    """
    Dual Attentive Fusion Module.

    Input:
        B × 22 × 640

    Internally:
        B × 22 × T × C

    The spatial and temporal attention vectors are
    combined by an outer product to create an
    interactive spatial-temporal attention map.

    A residual path preserves the original features.
    """

    def __init__(
        self,
        n_channels=22,
        hidden_channels=8,
        dropout=0.20,
    ):

        super().__init__()

        self.n_channels = (
            n_channels
        )

        self.feature = nn.Conv2d(
            1,
            hidden_channels,
            kernel_size=(
                3,
                7,
            ),
            padding=(
                1,
                3,
            ),
            bias=False,
        )

        self.bn = nn.BatchNorm2d(
            hidden_channels
        )

        # Spatial attention:
        # B × C × H × W
        # -> B × H × hidden_channels
        self.spatial_proj = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(
                1,
                1,
            ),
            bias=True,
        )

        # Temporal attention:
        # B × C × H × W
        # -> B × W × hidden_channels
        self.temporal_proj = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(
                1,
                1,
            ),
            bias=True,
        )

        self.residual = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(
                1,
                1,
            ),
            bias=False,
        )

        self.out_norm = nn.BatchNorm2d(
            hidden_channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):

        # x = B,C,T
        x2 = x.unsqueeze(
            1
        )

        B = x2.shape[0]
        H = x2.shape[2]
        W = x2.shape[3]

        Bfeat = self.feature(
            x2
        )

        Bfeat = self.bn(
            Bfeat
        )

        Bfeat = F.elu(
            Bfeat
        )

        # ----------------------------------------------------
        # Spatial attention
        # ----------------------------------------------------

        spatial_logits = (
            self.spatial_proj(
                Bfeat
            )
            .mean(
                dim=3
            )
            .transpose(
                1,
                2,
            )
        )

        # B,H,F
        spatial_attn = torch.softmax(
            spatial_logits,
            dim=1,
        )

        # ----------------------------------------------------
        # Temporal attention
        # ----------------------------------------------------

        temporal_logits = (
            self.temporal_proj(
                Bfeat
            )
            .mean(
                dim=2
            )
            .transpose(
                1,
                2,
            )
        )

        # B,W,F
        temporal_attn = torch.softmax(
            temporal_logits,
            dim=1,
        )

        # ----------------------------------------------------
        # Interactive spatial-temporal fusion
        # ----------------------------------------------------
        #
        # B,H,F × B,W,F
        # -> B,H,W
        #
        attention_map = torch.einsum(
            "bhf,bwf->bhw",
            spatial_attn,
            temporal_attn,
        )

        # normalize attention map over H×W
        attention_map = (
            attention_map
            /
            (
                attention_map.sum(
                    dim=(1,2),
                    keepdim=True,
                )
                + 1e-8
            )
        )

        attention_map = (
            attention_map
            .unsqueeze(1)
        )

        fused = (
            Bfeat
            * (
                1.0
                + attention_map
                * float(H * W)
            )
        )

        fused = (
            fused
            + self.residual(
                Bfeat
            )
        )

        fused = self.out_norm(
            fused
        )

        fused = F.elu(
            fused
        )

        fused = self.dropout(
            fused
        )

        return fused


print(
    "✅ DAFM attention block ready."
)

✅ DAFM attention block ready.


In [8]:
# ============================================================
# CELL 8 — EEGNET-STYLE FEATURE CLASSIFIER
# ============================================================

class DAFMFeatureClassifier(
    nn.Module
):

    def __init__(
        self,
        feature_channels=8,
        n_channels=22,
        n_classes=3,
        K1=32,
        K2=16,
        D=2,
        dropout=0.35,
    ):

        super().__init__()

        F2 = (
            feature_channels
            * D
        )

        # Temporal filtering.
        self.temporal = nn.Sequential(

            nn.Conv2d(
                feature_channels,
                feature_channels,
                kernel_size=(
                    1,
                    K1,
                ),
                padding=(
                    0,
                    K1 // 2,
                ),
                groups=feature_channels,
                bias=False,
            ),

            nn.BatchNorm2d(
                feature_channels
            ),

            nn.ELU(),
        )

        # Spatial filtering.
        self.spatial = nn.Sequential(

            nn.Conv2d(
                feature_channels,
                F2,
                kernel_size=(
                    n_channels,
                    1,
                ),
                groups=feature_channels,
                bias=False,
            ),

            nn.BatchNorm2d(
                F2
            ),

            nn.ELU(),

            nn.AvgPool2d(
                (
                    1,
                    4,
                )
            ),

            nn.Dropout(
                dropout
            ),
        )

        # Separable temporal refinement.
        self.sep_depth = nn.Conv2d(
            F2,
            F2,
            kernel_size=(
                1,
                K2,
            ),
            padding=(
                0,
                K2 // 2,
            ),
            groups=F2,
            bias=False,
        )

        self.sep_point = nn.Conv2d(
            F2,
            F2,
            kernel_size=(
                1,
                1,
            ),
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F2
        )

        self.pool2 = nn.AvgPool2d(
            (
                1,
                4,
            )
        )

        self.drop2 = nn.Dropout(
            dropout
        )

        # Dynamically determine flattened dimension.
        with torch.no_grad():

            dummy = torch.zeros(
                2,
                feature_channels,
                n_channels,
                640,
            )

            z = self.temporal(
                dummy
            )

            z = self.spatial(
                z
            )

            z = self.sep_depth(
                z
            )

            z = self.sep_point(
                z
            )

            z = self.bn2(
                z
            )

            z = F.elu(
                z
            )

            z = self.pool2(
                z
            )

            z = self.drop2(
                z
            )

            flat_dim = int(
                np.prod(
                    z.shape[1:]
                )
            )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                flat_dim,
                64,
            ),

            nn.ELU(),

            nn.Dropout(
                0.40
            ),

            nn.Linear(
                64,
                n_classes,
            ),
        )

    def forward(
        self,
        z,
    ):

        z = self.temporal(
            z
        )

        z = self.spatial(
            z
        )

        z = self.sep_depth(
            z
        )

        z = self.sep_point(
            z
        )

        z = self.bn2(
            z
        )

        z = F.elu(
            z
        )

        z = self.pool2(
            z
        )

        z = self.drop2(
            z
        )

        return self.classifier(
            z
        )


class DAFMNet3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_samples=640,
        n_classes=3,
    ):

        super().__init__()

        self.dafm = DAFM3C(
            n_channels=n_channels,
            hidden_channels=8,
            dropout=0.20,
        )

        self.classifier = (
            DAFMFeatureClassifier(
                feature_channels=8,
                n_channels=n_channels,
                n_classes=n_classes,
                K1=32,
                K2=16,
                D=2,
                dropout=0.35,
            )
        )

    def forward(
        self,
        x,
    ):

        z = self.dafm(
            x
        )

        return self.classifier(
            z
        )


# ------------------------------------------------------------
# Forward test
# ------------------------------------------------------------

_test_model = DAFMNet3C().to(
    device
)

with torch.no_grad():

    dummy = torch.randn(
        4,
        22,
        640,
        device=device,
    )

    logits = _test_model(
        dummy
    )

print(
    "Output shape:",
    tuple(
        logits.shape
    )
)

print(
    "Parameters:",
    f"{sum(p.numel() for p in _test_model.parameters() if p.requires_grad):,}"
)

assert tuple(
    logits.shape
) == (
    4,
    3,
)

print(
    "✅ DAFM-3C forward PASS."
)

Output shape: (4, 3)
Parameters: 42,827
✅ DAFM-3C forward PASS.


In [9]:
# ============================================================
# CELL 9 — EEG AUGMENTATION + BALANCED DATA LOADER
# ============================================================

def augment_subject_aware(
    x,
):
    """
    Conservative source training augmentation.
    """

    x = x.clone()

    B, C, T = x.shape

    # Amplitude jitter
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.40:

        x = (
            x
            * torch.empty(
                B, 1, 1,
                device=x.device,
            ).uniform_(
                0.90,
                1.10,
            )
        )

    # Sensor noise
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.25:

        x = (
            x
            + 0.004
            * torch.randn_like(
                x
            )
        )

    # Small temporal translation
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.20:

        shifts = torch.randint(
            -8,
            9,
            (B,),
            device=x.device,
        )

        x = torch.stack(
            [
                torch.roll(
                    x[i],
                    int(
                        shifts[i].item()
                    ),
                    dims=-1,
                )
                for i in range(B)
            ],
            dim=0,
        )

    return x


def make_train_loader(
    X,
    y,
    batch_size=64,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    dataset = TensorDataset(
        torch.from_numpy(
            X
        ),
        torch.from_numpy(
            y
        ),
    )

    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(
        np.float64
    )

    inv = np.zeros_like(
        counts
    )

    valid = counts > 0

    inv[valid] = (
        1.0
        / counts[valid]
    )

    weights = inv[y]

    sampler = (
        WeightedRandomSampler(
            torch.as_tensor(
                weights,
                dtype=torch.double,
            ),
            num_samples=len(y),
            replacement=True,
        )
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


def make_eval_loader(
    X,
    batch_size=128,
):

    dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )


print(
    "✅ Training/evaluation loaders ready."
)

✅ Training/evaluation loaders ready.


In [10]:
# ============================================================
# CELL 10 — TRAINING / PREDICTION
# ============================================================

@torch.no_grad()
def predict_dafm(
    model,
    X,
):

    model.eval()

    loader = make_eval_loader(
        X
    )

    all_logits = []

    for xb, _ in loader:

        xb = xb.to(
            device,
            non_blocking=True,
        )

        logits = model(
            xb
        )

        all_logits.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        all_logits,
        axis=0,
    )

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    P = np.exp(
        logits
    )

    P /= (
        P.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return P.astype(
        np.float32
    )


def train_dafm_one_seed(
    X_train,
    y_train,
    X_val,
    y_val,
    seed,
    epochs=120,
    batch_size=64,
    lr=7e-4,
    patience=20,
):

    seed_everything(
        seed
    )

    model = DAFMNet3C().to(
        device
    )

    counts = np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).astype(
        np.float32
    )

    class_weights = (
        counts.sum()
        /
        (
            N_CLASSES
            *
            np.maximum(
                counts,
                1.0,
            )
        )
    )

    class_weights /= (
        class_weights.mean()
        + 1e-12
    )

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=device,
        ),
        label_smoothing=0.015,
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=3e-4,
        betas=(
            0.9,
            0.999,
        ),
    )

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.80,
            patience=7,
            min_lr=1e-5,
        )
    )

    loader = make_train_loader(
        X_train,
        y_train,
        batch_size=batch_size,
    )

    best_state = None
    best_loss = np.inf
    best_bacc = -np.inf
    best_epoch = 0
    wait = 0

    history = []

    print(
        "\n"
        + "-" * 72
    )

    print(
        f"DAFM seed = {seed}"
    )

    print(
        "-" * 72
    )

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        losses = []

        train_true = []
        train_pred = []

        for xb, yb in loader:

            xb = xb.to(
                device,
                non_blocking=True,
            )

            yb = yb.to(
                device,
                non_blocking=True,
            )

            xb = augment_subject_aware(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb,
            )

            if not torch.isfinite(
                loss
            ):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=2.0,
            )

            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            train_true.extend(
                yb.detach()
                .cpu()
                .numpy()
            )

            train_pred.extend(
                logits.argmax(
                    dim=1
                )
                .detach()
                .cpu()
                .numpy()
            )

        P_val = predict_dafm(
            model,
            X_val,
        )

        pred_val = (
            P_val.argmax(
                axis=1
            )
        )

        val_loss = -float(
            np.mean(
                np.log(
                    np.clip(
                        P_val[
                            np.arange(
                                len(y_val)
                            ),
                            y_val,
                        ],
                        1e-8,
                        1.0,
                    )
                )
            )
        )

        train_acc = (
            accuracy_score(
                train_true,
                train_pred,
            )
            * 100.0
        )

        val_acc = (
            accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        scheduler.step(
            val_loss
        )

        history.append(
            {
                "epoch":
                    epoch,
                "loss":
                    float(
                        np.mean(
                            losses
                        )
                    ),
                "val_loss":
                    val_loss,
                "train_acc":
                    train_acc,
                "val_acc":
                    val_acc,
                "val_bacc":
                    val_bacc,
                "lr":
                    optimizer.param_groups[
                        0
                    ]["lr"],
            }
        )

        if val_loss < (
            best_loss
            - 1e-5
        ):

            best_loss = (
                val_loss
            )

            best_bacc = (
                val_bacc
            )

            best_epoch = (
                epoch
            )

            wait = 0

            best_state = copy.deepcopy(
                model.state_dict()
            )

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"train={train_acc:5.1f}% | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"vLoss={val_loss:.4f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

        if wait >= patience:

            print(
                f"    early stop at "
                f"{epoch}; best={best_epoch}"
            )

            break

    if best_state is None:

        raise RuntimeError(
            "No valid DAFM checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
        best_epoch,
        best_loss,
        best_bacc,
    )


print(
    "✅ DAFM trainer ready."
)

✅ DAFM trainer ready.


In [11]:
# ============================================================
# CELL 11 — ONE DAFM LOSO FOLD
# ============================================================

def run_dafm_fold(
    target_subject,
    fold_id,
    n_seeds=2,
    epochs=120,
    batch_size=64,
    lr=7e-4,
    patience=20,
):

    t0 = time.time()

    target_subject = str(
        target_subject
    )

    # --------------------------------------------------------
    # Outer LOSO metadata
    # --------------------------------------------------------

    source_mask = (
        bci_meta[
            "subject"
        ].astype(str)
        != target_subject
    )

    target_mask = (
        bci_meta[
            "subject"
        ].astype(str)
        == target_subject
    )

    source_idx = (
        bci_meta.loc[
            source_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    target_idx = (
        bci_meta.loc[
            target_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X_source_raw = load_indices(
        source_idx
    )

    X_test_raw = load_indices(
        target_idx
    )

    y_source = (
        bci_meta.loc[
            source_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_test = (
        bci_meta.loc[
            target_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    # --------------------------------------------------------
    # Source-only normalization
    # --------------------------------------------------------

    normalizer = (
        SourceRobustNormalizer()
        .fit(
            X_source_raw
        )
    )

    X_source = (
        normalizer.transform(
            X_source_raw
        )
    )

    X_test = (
        normalizer.transform(
            X_test_raw
        )
    )

    # --------------------------------------------------------
    # Stratified source trial validation
    # --------------------------------------------------------

    train_local, val_local = (
        stratified_split(
            len(
                X_source
            ),
            y_source,
            val_fraction=0.20,
            seed=SEED,
        )
    )

    X_train = (
        X_source[
            train_local
        ]
    )

    y_train = (
        y_source[
            train_local
        ]
    )

    X_val = (
        X_source[
            val_local
        ]
    )

    y_val = (
        y_source[
            val_local
        ]
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"DAFM-3C LOSO "
        f"[{fold_id}/9] — target {target_subject}"
    )

    print(
        "=" * 78
    )

    print(
        "Train:",
        X_train.shape,
    )

    print(
        "Val  :",
        X_val.shape,
    )

    print(
        "Test :",
        X_test.shape,
    )

    # --------------------------------------------------------
    # Multi-seed ensemble
    # --------------------------------------------------------

    models = []
    seed_rows = []
    histories = []

    for k in range(
        n_seeds
    ):

        model_seed = (
            42
            if k == 0
            else 123
        )

        (
            model,
            history,
            best_epoch,
            best_loss,
            best_bacc,
        ) = train_dafm_one_seed(
            X_train,
            y_train,
            X_val,
            y_val,
            seed=model_seed,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
        )

        models.append(
            model
        )

        histories.append(
            history
        )

        seed_rows.append(
            {
                "seed":
                    model_seed,
                "best_epoch":
                    best_epoch,
                "best_val_loss":
                    best_loss,
                "best_val_bacc":
                    best_bacc,
            }
        )

    seed_summary = pd.DataFrame(
        seed_rows
    )

    print(
        "\nSeed summary:"
    )

    display(
        seed_summary
    )

    # --------------------------------------------------------
    # Validation ensemble
    # --------------------------------------------------------

    P_val = np.mean(
        np.stack(
            [
                predict_dafm(
                    model,
                    X_val,
                )
                for model in models
            ],
            axis=0,
        ),
        axis=0,
    )

    pred_val = (
        P_val.argmax(
            axis=1
        )
    )

    val_acc = (
        accuracy_score(
            y_val,
            pred_val,
        )
        * 100.0
    )

    val_bacc = (
        balanced_accuracy_score(
            y_val,
            pred_val,
        )
        * 100.0
    )

    # --------------------------------------------------------
    # External target prediction
    # --------------------------------------------------------

    P_test = np.mean(
        np.stack(
            [
                predict_dafm(
                    model,
                    X_test,
                )
                for model in models
            ],
            axis=0,
        ),
        axis=0,
    )

    pred_test = (
        P_test.argmax(
            axis=1
        )
    )

    test_acc = (
        accuracy_score(
            y_test,
            pred_test,
        )
        * 100.0
    )

    test_bacc = (
        balanced_accuracy_score(
            y_test,
            pred_test,
        )
        * 100.0
    )

    kappa = (
        cohen_kappa_score(
            y_test,
            pred_test,
        )
    )

    elapsed = (
        time.time()
        - t0
    )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Target subject       : "
        f"{target_subject}"
    )

    print(
        f"Validation ensemble  : "
        f"{val_bacc:.2f}%"
    )

    print(
        f"External test acc    : "
        f"{test_acc:.2f}%"
    )

    print(
        f"External test bAcc   : "
        f"{test_bacc:.2f}%"
    )

    print(
        f"Kappa                : "
        f"{kappa:.4f}"
    )

    print(
        f"Elapsed              : "
        f"{elapsed / 60.0:.1f} min"
    )

    print(
        "-" * 78
    )

    return {
        "fold":
            fold_id,
        "subject":
            target_subject,
        "test_acc":
            float(
                test_acc
            ),
        "test_bacc":
            float(
                test_bacc
            ),
        "val_acc":
            float(
                val_acc
            ),
        "val_bacc":
            float(
                val_bacc
            ),
        "kappa":
            float(
                kappa
            ),
        "y_test":
            y_test,
        "pred_test":
            pred_test,
        "P_test":
            P_test,
        "models":
            models,
        "histories":
            histories,
        "seed_summary":
            seed_summary,
    }


print(
    "✅ DAFM LOSO fold ready."
)

✅ DAFM LOSO fold ready.


In [12]:
# ============================================================
# CELL 12 — S01 SMOKE TEST
# ============================================================

smoke_result = run_dafm_fold(
    target_subject="S01",
    fold_id=1,
    n_seeds=2,
    epochs=120,
    batch_size=64,
    lr=7e-4,
    patience=20,
)

print(
    "\n"
    + "=" * 78
)

print(
    "DAFM-3C S01 SMOKE TEST"
)

print(
    "=" * 78
)

print(
    f"S01 accuracy : "
    f"{smoke_result['test_acc']:.2f}%"
)

print(
    f"S01 bAcc     : "
    f"{smoke_result['test_bacc']:.2f}%"
)

print(
    f"S01 kappa    : "
    f"{smoke_result['kappa']:.4f}"
)


DAFM-3C LOSO [1/9] — target S01
Train: (1383, 22, 640)
Val  : (345, 22, 640)
Test : (216, 22, 640)

------------------------------------------------------------------------
DAFM seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 31.9% | val= 39.1% | bAcc= 39.1% | vLoss=1.0914 | lr=7.00e-04
    epoch 010 | train= 58.1% | val= 53.0% | bAcc= 53.0% | vLoss=0.9304 | lr=7.00e-04
    epoch 020 | train= 63.2% | val= 56.2% | bAcc= 56.2% | vLoss=0.8620 | lr=7.00e-04
    epoch 030 | train= 66.5% | val= 59.4% | bAcc= 59.4% | vLoss=0.8153 | lr=7.00e-04
    epoch 040 | train= 67.8% | val= 58.3% | bAcc= 58.3% | vLoss=0.8238 | lr=7.00e-04
    epoch 050 | train= 70.9% | val= 59.4% | bAcc= 59.4% | vLoss=0.8188 | lr=5.60e-04
    epoch 060 | train= 71.4% | val= 60.0% | bAcc= 60.0% | vLoss=0.8088 | lr=4.48e-04
    early stop at 61; best=41

------------------------------------------------------------------------
DAFM seed = 123
-----------------------

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,41,0.792147,61.449275
1,123,97,0.758914,62.898551



------------------------------------------------------------------------------
Target subject       : S01
Validation ensemble  : 65.51%
External test acc    : 74.54%
External test bAcc   : 74.54%
Kappa                : 0.6181
Elapsed              : 31.5 min
------------------------------------------------------------------------------

DAFM-3C S01 SMOKE TEST
S01 accuracy : 74.54%
S01 bAcc     : 74.54%
S01 kappa    : 0.6181


In [14]:
# ============================================================
# CELL 13 — FULL 9-SUBJECT LOSO
# ============================================================

RUN_FULL_LOSO = True

if RUN_FULL_LOSO:

    all_results = []
    fold_objects = {}

    for fold_id, subject in enumerate(
        BCI_SUBJECTS,
        start=1,
    ):

        result = run_dafm_fold(
            target_subject=subject,
            fold_id=fold_id,
            n_seeds=2,
            epochs=120,
            batch_size=64,
            lr=7e-4,
            patience=20,
        )

        fold_objects[
            subject
        ] = result

        all_results.append(
            {
                "fold":
                    fold_id,
                "subject":
                    subject,
                "val_bacc":
                    result[
                        "val_bacc"
                    ],
                "test_accuracy":
                    result[
                        "test_acc"
                    ],
                "test_bacc":
                    result[
                        "test_bacc"
                    ],
                "kappa":
                    result[
                        "kappa"
                    ],
            }
        )

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

    dafm_results_df = pd.DataFrame(
        all_results
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "DAFM-3C — BCI-IV-2a LOSO SUMMARY"
    )

    print(
        "=" * 78
    )

    display(
        dafm_results_df
    )

    mean_acc = (
        dafm_results_df[
            "test_accuracy"
        ].mean()
    )

    mean_bacc = (
        dafm_results_df[
            "test_bacc"
        ].mean()
    )

    median_acc = (
        dafm_results_df[
            "test_accuracy"
        ].median()
    )

    std_acc = (
        dafm_results_df[
            "test_accuracy"
        ].std()
    )

    n70 = int(
        (
            dafm_results_df[
                "test_accuracy"
            ]
            >= 70.0
        ).sum()
    )

    n80 = int(
        (
            dafm_results_df[
                "test_accuracy"
            ]
            >= 80.0
        ).sum()
    )

    print(
        f"\nMean accuracy : {mean_acc:.2f}%"
    )

    print(
        f"Mean bAcc     : {mean_bacc:.2f}%"
    )

    print(
        f"Median        : {median_acc:.2f}%"
    )

    print(
        f"Std           : {std_acc:.2f}%"
    )

    print(
        f"Subjects >=70 : {n70}/9"
    )

    print(
        f"Subjects >=80 : {n80}/9"
    )

    if mean_acc >= 80.0:

        print(
            "\n✅ 80% TARGET ACHIEVED."
        )

    elif mean_acc >= 70.0:

        print(
            "\n✅ 70% TARGET ACHIEVED."
        )

    else:

        print(
            "\n❌ Target not yet achieved."
        )


DAFM-3C LOSO [1/9] — target S01
Train: (1383, 22, 640)
Val  : (345, 22, 640)
Test : (216, 22, 640)

------------------------------------------------------------------------
DAFM seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 31.9% | val= 39.1% | bAcc= 39.1% | vLoss=1.0914 | lr=7.00e-04
    epoch 010 | train= 58.1% | val= 53.0% | bAcc= 53.0% | vLoss=0.9304 | lr=7.00e-04
    epoch 020 | train= 63.2% | val= 56.2% | bAcc= 56.2% | vLoss=0.8620 | lr=7.00e-04
    epoch 030 | train= 66.5% | val= 59.4% | bAcc= 59.4% | vLoss=0.8153 | lr=7.00e-04
    epoch 040 | train= 67.8% | val= 58.3% | bAcc= 58.3% | vLoss=0.8238 | lr=7.00e-04
    epoch 050 | train= 70.9% | val= 59.4% | bAcc= 59.4% | vLoss=0.8188 | lr=5.60e-04
    epoch 060 | train= 71.4% | val= 60.0% | bAcc= 60.0% | vLoss=0.8088 | lr=4.48e-04
    early stop at 61; best=41

------------------------------------------------------------------------
DAFM seed = 123
-----------------------

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,41,0.792147,61.449275
1,123,97,0.758914,62.898551



------------------------------------------------------------------------------
Target subject       : S01
Validation ensemble  : 65.51%
External test acc    : 74.54%
External test bAcc   : 74.54%
Kappa                : 0.6181
Elapsed              : 31.4 min
------------------------------------------------------------------------------

DAFM-3C LOSO [2/9] — target S02
Train: (1383, 22, 640)
Val  : (345, 22, 640)
Test : (216, 22, 640)

------------------------------------------------------------------------
DAFM seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 32.2% | val= 41.2% | bAcc= 41.2% | vLoss=1.0886 | lr=7.00e-04
    epoch 010 | train= 60.5% | val= 58.6% | bAcc= 58.6% | vLoss=0.8643 | lr=7.00e-04
    epoch 020 | train= 66.7% | val= 62.3% | bAcc= 62.3% | vLoss=0.8083 | lr=7.00e-04
    epoch 030 | train= 69.9% | val= 67.2% | bAcc= 67.2% | vLoss=0.7630 | lr=7.00e-04
    epoch 040 | train= 70.8% | val= 64.6% | bAcc= 64.6% | vL

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,43,0.737856,66.666667
1,123,73,0.718197,67.536232



------------------------------------------------------------------------------
Target subject       : S02
Validation ensemble  : 68.99%
External test acc    : 37.96%
External test bAcc   : 37.96%
Kappa                : 0.0694
Elapsed              : 28.3 min
------------------------------------------------------------------------------

DAFM-3C LOSO [3/9] — target S03
Train: (1383, 22, 640)
Val  : (345, 22, 640)
Test : (216, 22, 640)

------------------------------------------------------------------------
DAFM seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 32.1% | val= 39.1% | bAcc= 39.1% | vLoss=1.0915 | lr=7.00e-04
    epoch 010 | train= 56.2% | val= 52.5% | bAcc= 52.5% | vLoss=0.9335 | lr=7.00e-04
    epoch 020 | train= 62.5% | val= 55.7% | bAcc= 55.7% | vLoss=0.8799 | lr=7.00e-04


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 14 — CONFUSION MATRIX + FINAL REPORT
# ============================================================

if (
    "fold_objects" in globals()
    and len(fold_objects) > 0
):

    y_all = np.concatenate(
        [
            fold_objects[s][
                "y_test"
            ]
            for s in BCI_SUBJECTS
        ]
    )

    pred_all = np.concatenate(
        [
            fold_objects[s][
                "pred_test"
            ]
            for s in BCI_SUBJECTS
        ]
    )

    cm = confusion_matrix(
        y_all,
        pred_all,
        labels=list(
            range(N_CLASSES)
        ),
        normalize="true",
    )

    print(
        "Normalized confusion matrix:"
    )

    display(
        pd.DataFrame(
            cm,
            index=CLASSES,
            columns=CLASSES,
        ).round(3)
    )

    print(
        "\nClassification report:"
    )

    print(
        classification_report(
            y_all,
            pred_all,
            labels=list(
                range(N_CLASSES)
            ),
            target_names=CLASSES,
            digits=4,
        )
    )

    plt.figure(
        figsize=(6,5)
    )

    plt.imshow(
        cm,
        interpolation="nearest",
    )

    plt.xticks(
        range(N_CLASSES),
        CLASSES,
    )

    plt.yticks(
        range(N_CLASSES),
        CLASSES,
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        "DAFM-3C — BCI-IV-2a LOSO"
    )

    for i in range(
        N_CLASSES
    ):

        for j in range(
            N_CLASSES
        ):

            plt.text(
                j,
                i,
                f"{cm[i,j]:.2f}",
                ha="center",
                va="center",
            )

    plt.tight_layout()

    fig_path = (
        FIG_DIR
        / "dafm_3c_loso_confusion_matrix.png"
    )

    plt.savefig(
        fig_path,
        dpi=180,
    )

    plt.show()

    print(
        "\nSaved:",
        fig_path,
    )

In [ ]:
# ============================================================
# CELL 15 — SAVE FINAL RESULTS
# ============================================================

if (
    "dafm_results_df" in globals()
):

    csv_path = (
        RESULT_DIR
        / "dafm_3c_bci_iv2a_loso_results.csv"
    )

    dafm_results_df.to_csv(
        csv_path,
        index=False,
    )

    print(
        "Results saved:",
        csv_path,
    )

protocol = {
    "model":
        "DAFM-3C",

    "architecture":
        "Dual spatial-temporal attentive fusion + EEGNet-style classifier",

    "dataset":
        "BCI-IV-2a",

    "subjects":
        BCI_SUBJECTS,

    "input_shape":
        [22, 640],

    "classes":
        CLASSES,

    "sampling_rate_hz":
        160,

    "bandpass_hz":
        [8, 30],

    "outer_evaluation":
        "9-subject LOSO",

    "normalization":
        "source-only robust channel normalization",

    "validation":
        "stratified 80/20 source-trial split",

    "ensemble":
        [42, 123],

    "target_labels_used_for_training":
        False,

    "target_labels_used_for_model_selection":
        False,
}

protocol_path = (
    RESULT_DIR
    / "dafm_3c_protocol.json"
)

with open(
    protocol_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )

print(
    "Protocol saved:",
    protocol_path,
)